## K-1. Menyiapkan Lingkungan dan Perkakas


In [3]:
import sys, time, os, json, sqlite3, gc, shutil, glob, warnings
import pandas as pd, numpy as np, psutil
warnings.filterwarnings("ignore", category=FutureWarning)

for nama in ["pandas", "numpy", "pyarrow", "requests", "sklearn", "sqlalchemy"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s}: {mod.__version__}")
    except ImportError:
        print(f"{nama:11s}: BELUM TERPASANG")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum3"
except Exception as e:
    print("Drive tidak tersedia, simpan lokal:", e)
    DIR_SIMPAN = "/content/hasil_praktikum3"

DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
for d in (DIR_MENTAH, DIR_KURASI, DIR_SIMPAN):
    os.makedirs(d, exist_ok=True)

# --- Dipakai kembali dari Praktikum 1 / K-3 ---
proses = psutil.Process(os.getpid())
catatan = []
def rss_mb():
    return proses.memory_info().rss / 1024**2

def ukur(label, fungsi):
    m0, t0 = rss_mb(), time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 2),
                    "delta_rss_mb": round(rss_mb() - m0, 1)})
    print(f"[{label}] {detik:.2f} s")
    return hasil

# --- Baru di Praktikum 3: catatan asal-usul data (lineage) ---
lineage = []
def catat_sumber(nama, asal, jumlah_baris, keterangan=""):
    lineage.append({
        "sumber": nama, "asal": asal,
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": jumlah_baris, "keterangan": keterangan,
    })
    print(f"[lineage] {nama}: {jumlah_baris:,} baris")

# --- Pelacak susut data (dipakai di analisis O-6 dan diagram lineage) ---
susut = []

pandas     : 2.2.3
numpy      : 2.1.3
pyarrow    : 23.0.1
requests   : 2.32.4
sklearn    : 1.6.1
sqlalchemy : 2.0.54
Mounted at /content/drive


## K-2. Akuisisi 1 — Unduhan Berkala yang Tahan Gagal


In [4]:
import requests

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    """Unduh dengan retry + penulisan atomik. Melewati unduhan bila file sudah ada."""
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan
    sementara = tujuan + ".part"
    for i in range(percobaan):
        try:
            t0 = time.perf_counter()
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    for potong in r.iter_content(chunk_size=1 << 20):
                        f.write(potong)
            ukuran = os.path.getsize(sementara)
            if ukuran == 0:
                raise IOError("berkas kosong")
            os.replace(sementara, tujuan)          # atomik
            detik = max(time.perf_counter() - t0, 1e-6)
            mb = ukuran / 1024**2
            print(f"unduh {os.path.basename(tujuan)}: {mb:.1f} MB dalam {detik:.1f}s ({mb/detik:.2f} MB/s)")
            return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i)             # exponential backoff
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)
    raise RuntimeError(f"gagal mengunduh setelah {percobaan} percobaan: {url}")

URL_TRIP = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
URL_ZONA = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
PATH_TRIP = os.path.join(DIR_MENTAH, "yellow_tripdata_2023-01.parquet")
PATH_ZONA = os.path.join(DIR_MENTAH, "taxi_zone_lookup.csv")

KOLOM = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
         "trip_distance", "PULocationID", "DOLocationID", "payment_type",
         "fare_amount", "tip_amount", "total_amount"]

SUMBER = {}   # "nyata" / "sintetis" per sumber, dilaporkan di lineage

def rentang_bulan(bulan):
    """'2023-01' -> (awal inklusif, akhir eksklusif)"""
    awal = pd.Timestamp(bulan + "-01")
    return awal, awal + pd.offsets.MonthBegin(1)

# ---- Jalur cadangan (K-4): pembangkit data sintetis ----
def trip_sintetis(n=500_000, seed=11, bulan="2023-01"):
    rng = np.random.default_rng(seed)
    awal, akhir = rentang_bulan(bulan)
    detik_bulan = int((akhir - awal).total_seconds())
    naik = awal + pd.to_timedelta(rng.integers(0, detik_bulan, n), unit="s")
    durasi = rng.gamma(2.0, 7.0, n)
    m_out = rng.random(n) < 0.005
    durasi[m_out] = rng.uniform(300, 900, int(m_out.sum()))
    turun = naik + pd.to_timedelta(durasi * 60, unit="s")
    jarak = np.round(rng.lognormal(0.7, 0.8, n), 2)
    jarak[rng.random(n) < 0.005] = 0.0
    pay = rng.choice([1, 2, 3, 4], n, p=[0.75, 0.2, 0.03, 0.02])
    fare = np.round(3 + 2.5 * jarak + 0.5 * durasi, 2)
    tip = np.where(pay == 1, np.round(fare * rng.uniform(0, 0.3, n), 2), 0.0)
    total = np.round(fare + tip + 3.5, 2)
    pax = rng.choice([1, 2, 3, 4, 5, 6], n, p=[0.7, 0.15, 0.05, 0.04, 0.04, 0.02]).astype(float)
    pax[rng.random(n) < 0.03] = np.nan
    df = pd.DataFrame({
        "tpep_pickup_datetime": naik, "tpep_dropoff_datetime": turun,
        "passenger_count": pax, "trip_distance": jarak,
        "PULocationID": rng.integers(1, 264, n), "DOLocationID": rng.integers(1, 264, n),
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip, "total_amount": total})
    return pd.concat([df, df.head(1000)], ignore_index=True)   # sisipkan duplikat

def zona_sintetis():
    rng = np.random.default_rng(3)
    boro = rng.choice(["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"],
                      265, p=[0.3, 0.25, 0.25, 0.15, 0.04, 0.01]).astype(object)
    boro[263] = "Unknown"
    boro[264] = np.nan
    return pd.DataFrame({"LocationID": np.arange(1, 266), "Borough": boro,
                         "Zone": [f"Zona {i}" for i in range(1, 266)], "service_zone": "Boro Zone"})

def pastikan_trip(bulan):
    """Unduh berkas bulanan; bila gagal, buat sintetis dan tulis ke path yang sama."""
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{bulan}.parquet"
    path = os.path.join(DIR_MENTAH, f"yellow_tripdata_{bulan}.parquet")
    try:
        unduh_aman(url, path)
        SUMBER.setdefault(f"trip_{bulan}", "nyata")
    except Exception as e:
        print(f"PERINGATAN: unduhan {bulan} gagal ({e}). Memakai data SINTETIS.")
        if not os.path.exists(path):
            trip_sintetis(bulan=bulan, seed=int(bulan[-2:])).to_parquet(path, index=False)
        SUMBER[f"trip_{bulan}"] = "sintetis"
    return path

def pastikan_zona():
    try:
        unduh_aman(URL_ZONA, PATH_ZONA)
        SUMBER["zona"] = "nyata"
    except Exception as e:
        print(f"PERINGATAN: unduhan zona gagal ({e}). Memakai zona SINTETIS.")
        if not os.path.exists(PATH_ZONA):
            zona_sintetis().to_csv(PATH_ZONA, index=False)
        SUMBER["zona"] = "sintetis"
    return PATH_ZONA

ukur("unduh trip", lambda: pastikan_trip("2023-01"))
ukur("unduh zona", pastikan_zona)
for pth in (PATH_TRIP, PATH_ZONA):
    print(os.path.basename(pth), "->", round(os.path.getsize(pth) / 1024**2, 2), "MB")

trip = ukur("baca trip", lambda: pd.read_parquet(PATH_TRIP, columns=KOLOM))
zona = pd.read_csv(PATH_ZONA)
catat_sumber("trip", URL_TRIP if SUMBER["trip_2023-01"] == "nyata" else "sintetis", len(trip),
             "Parquet bulanan TLC" if SUMBER["trip_2023-01"] == "nyata" else "PENGGANTI - bukan data nyata")
catat_sumber("zona", URL_ZONA if SUMBER["zona"] == "nyata" else "sintetis", len(zona),
             "tabel dimensi 265 zona")
zona.head()

unduh yellow_tripdata_2023-01.parquet: 45.5 MB dalam 0.3s (156.39 MB/s)
[unduh trip] 0.29 s
unduh taxi_zone_lookup.csv: 0.0 MB dalam 0.0s (0.34 MB/s)
[unduh zona] 0.04 s
yellow_tripdata_2023-01.parquet -> 45.46 MB
taxi_zone_lookup.csv -> 0.01 MB
[baca trip] 1.54 s
[lineage] trip: 3,066,766 baris
[lineage] zona: 265 baris


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


## K-3. Akuisisi 2 — API JSON (Open-Meteo)


In [5]:
API_CUACA = "https://archive-api.open-meteo.com/v1/archive"

def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060, percobaan=3):
    parameter = {
        "latitude": lat, "longitude": lon,
        "start_date": mulai, "end_date": selesai,
        "hourly": "temperature_2m,precipitation",
        "timezone": "America/New_York",
    }
    for i in range(percobaan):
        try:
            r = requests.get(API_CUACA, params=parameter, timeout=60)
        except requests.RequestException as e:      # jaringan: layak dicoba ulang
            print(f"percobaan {i+1}: {e}")
            time.sleep(2 ** i)
            continue
        if r.status_code == 200:
            data = r.json()
            if "hourly" not in data:                # validasi struktur
                raise ValueError("kunci 'hourly' tidak ada pada respons")
            return pd.DataFrame(data["hourly"])
        if r.status_code in (429, 500, 502, 503):   # layak dicoba ulang
            time.sleep(2 ** i)
            continue
        r.raise_for_status()                        # galat lain: hentikan
    raise RuntimeError("API cuaca tidak merespons dengan benar")

def rapikan_cuaca(df):
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"])
    df = df.rename(columns={"time": "jam_mulai", "temperature_2m": "suhu_c",
                            "precipitation": "hujan_mm"})
    df["jam_mulai"] = df["jam_mulai"].astype("datetime64[ns]")
    return df

def cuaca_sintetis(mulai="2023-01-01", selesai="2023-01-31", seed=7):
    jam = pd.date_range(mulai, pd.Timestamp(selesai) + pd.Timedelta("23h"), freq="h")
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "jam_mulai": jam.astype("datetime64[ns]"),
        "suhu_c": np.round(rng.normal(3, 5, len(jam)), 1),
        "hujan_mm": np.round(rng.gamma(0.4, 0.8, len(jam)), 2),
    })

def cuaca_bulan(bulan):
    """Cuaca satu bulan penuh -> (DataFrame, 'nyata'|'sintetis')"""
    awal, akhir = rentang_bulan(bulan)
    mulai, selesai = awal.strftime("%Y-%m-%d"), (akhir - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    try:
        return rapikan_cuaca(ambil_cuaca(mulai, selesai)), "nyata"
    except Exception as e:
        print(f"PERINGATAN: API cuaca gagal ({e}). Memakai cuaca SINTETIS.")
        return cuaca_sintetis(mulai, selesai), "sintetis"

(cuaca, SUMBER["cuaca_2023-01"]) = ukur("ambil cuaca", lambda: cuaca_bulan("2023-01"))
catat_sumber("cuaca", API_CUACA if SUMBER["cuaca_2023-01"] == "nyata" else "sintetis", len(cuaca),
             "Open-Meteo hourly archive" if SUMBER["cuaca_2023-01"] == "nyata" else "PENGGANTI - bukan data nyata")

jam_diharapkan = 31 * 24
print("baris cuaca:", len(cuaca), "| jam diharapkan:", jam_diharapkan,
      "| selisih:", jam_diharapkan - len(cuaca))
print("nilai hilang:\n", cuaca.isna().sum())
cuaca.head()

[ambil cuaca] 0.65 s
[lineage] cuaca: 744 baris
baris cuaca: 744 | jam diharapkan: 744 | selisih: 0
nilai hilang:
 jam_mulai    0
suhu_c       0
hujan_mm     0
dtype: int64


,jam_mulai,suhu_c,hujan_mm
0,2023-01-01 00:00:00,10.9,1.0
1,2023-01-01 01:00:00,10.6,1.0
2,2023-01-01 02:00:00,10.6,0.1
3,2023-01-01 03:00:00,10.5,0.0
4,2023-01-01 04:00:00,9.8,0.0


## K-4. Jalur Cadangan bila Internet Diblokir


In [6]:
print("Jalur akuisisi yang dipakai:")
for k, v in SUMBER.items():
    print(f"  {k:16s}: {v}")
if "sintetis" in SUMBER.values():
    print("\n>>> Ada sumber SINTETIS. Cantumkan hal ini di laporan; angka bukan data TLC/Open-Meteo asli.")

Jalur akuisisi yang dipakai:
  trip_2023-01    : nyata
  zona            : nyata
  cuaca_2023-01   : nyata


## K-5. Akuisisi 3 — SQLite dengan Pemuatan Bertahap


In [7]:
from sqlalchemy import create_engine

PATH_DB = os.path.join(DIR_MENTAH, "operasional.db")
engine = create_engine(f"sqlite:///{PATH_DB}")

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": ["Kartu kredit", "Tunai", "Gratis",
                        "Sengketa", "Tidak diketahui", "Perjalanan batal"],
    "kena_biaya_admin": [1, 0, 0, 0, 0, 0],
})
tarif_referensi.to_sql("tarif_referensi", engine, if_exists="replace", index=False)

trip.head(300_000).to_sql("trip_operasional", engine, if_exists="replace",
                          index=False, chunksize=50_000)
print("Tabel dibuat:", pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", engine)["name"].tolist())

SQL = """
    SELECT payment_type, COUNT(*) AS jumlah, AVG(total_amount) AS rata_total
    FROM trip_operasional
    WHERE total_amount > 0
    GROUP BY payment_type
    ORDER BY jumlah DESC
"""
ringkas_db = pd.read_sql(SQL, engine)
print(ringkas_db)

total_baris = 0
for bagian in pd.read_sql("SELECT trip_distance FROM trip_operasional",
                          engine, chunksize=50_000):
    total_baris += len(bagian)
print("dibaca bertahap:", f"{total_baris:,}", "baris")

tarif_referensi = pd.read_sql("SELECT * FROM tarif_referensi", engine)
catat_sumber("tarif_referensi", PATH_DB, len(tarif_referensi),
             "tabel dimensi dari basis data operasional")

Tabel dibuat: ['tarif_referensi', 'trip_operasional']
   payment_type  jumlah  rata_total
0             1  226967   31.339172
1             2   66504   25.218836
2             4    2082   26.611114
3             3    1553   22.089691
dibaca bertahap: 300,000 baris
[lineage] tarif_referensi: 6 baris


## K-6. Profiling Kualitas Data pada Enam Dimensi


In [8]:
trip["durasi_menit"] = ((trip["tpep_dropoff_datetime"] - trip["tpep_pickup_datetime"])
                        .dt.total_seconds() / 60)

def profil_kolom(df):
    baris = []
    for kol in df.columns:
        s = df[kol]
        baris.append({
            "kolom": kol, "tipe": str(s.dtype),
            "persen_hilang": round(100 * s.isna().mean(), 3),
            "nilai_unik": s.nunique(dropna=True),
            "contoh": s.dropna().iloc[0] if s.notna().any() else None,
        })
    return pd.DataFrame(baris)

profil = profil_kolom(trip)
display(profil)

AWAL, AKHIR = pd.Timestamp("2023-01-01"), pd.Timestamp("2023-02-01")
zona_sah = set(zona["LocationID"])

def hitung_laporan(df, awal, akhir):
    durasi = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    aturan = {
        "kelengkapan: passenger_count ada": df["passenger_count"].notna(),
        "keunikan: baris tidak duplikat": ~df.duplicated(),
        "validitas: jarak 0-100 mil": df["trip_distance"].between(0.01, 100),
        "validitas: total_amount > 0": df["total_amount"] > 0,
        "validitas: PULocationID dikenal": df["PULocationID"].isin(zona_sah),
        "konsistensi: durasi 1-180 menit": durasi.between(1, 180),
        "konsistensi: total >= fare": df["total_amount"] >= df["fare_amount"],
        "ketepatan waktu: dalam periode": (df["tpep_pickup_datetime"] >= awal)
                                          & (df["tpep_pickup_datetime"] < akhir),
    }
    lap = pd.DataFrame([
        {"aturan": nama, "lulus": int(m.sum()), "gagal": int((~m).sum()),
         "persen_gagal": round(100 * (~m).mean(), 3)}
        for nama, m in aturan.items()
    ]).sort_values("persen_gagal", ascending=False).reset_index(drop=True)
    return lap

laporan = hitung_laporan(trip, AWAL, AKHIR)
laporan

,kolom,tipe,persen_hilang,nilai_unik,contoh
0,tpep_pickup_datetime,datetime64[us],0.000,1610975,2023-01-01 00:32:10
1,tpep_dropoff_datetime,datetime64[us],0.000,1611319,2023-01-01 00:40:36
2,passenger_count,float64,2.339,10,1.0
3,trip_distance,float64,0.000,4387,0.97
4,PULocationID,int64,0.000,257,161
5,DOLocationID,int64,0.000,261,141
6,payment_type,int64,0.000,5,2
7,fare_amount,float64,0.000,6873,9.3
8,tip_amount,float64,0.000,4036,0.0
9,total_amount,float64,0.000,15871,14.3


,aturan,lulus,gagal,persen_gagal
0,kelengkapan: passenger_count ada,2995023,71743,2.339
1,validitas: jarak 0-100 mil,3020816,45950,1.498
2,konsistensi: durasi 1-180 menit,3030383,36383,1.186
3,validitas: total_amount > 0,3040994,25772,0.840
4,konsistensi: total >= fare,3041743,25023,0.816
5,ketepatan waktu: dalam periode,3066718,48,0.002
6,keunikan: baris tidak duplikat,3066766,0,0.000
7,validitas: PULocationID dikenal,3066766,0,0.000


## K-7. Nilai Hilang dan Duplikat


In [9]:
susut.append(("awal (mentah)", len(trip)))
total_sebelum = trip["total_amount"].copy()      # untuk histogram sebelum/sesudah (O)

KOL = "passenger_count"
asli = trip[KOL].copy()
print("persen hilang:", round(100 * asli.isna().mean(), 3))
trip["jam"] = trip["tpep_pickup_datetime"].dt.hour

strategi = {
    "0_asli(dengan NaN)": asli,
    "1_dibuang": asli.dropna(),
    "2_isi_median": asli.fillna(asli.median()),
    "3_isi_modus": asli.fillna(asli.mode().iloc[0]),
    "4_median_perjam": asli.fillna(trip.groupby("jam")[KOL].transform("median")),
}
banding = pd.DataFrame([
    {"strategi": nama, "n": len(s), "mean": round(s.mean(), 4),
     "median": s.median(), "std": round(s.std(), 4)}
    for nama, s in strategi.items()
])
display(banding)

# Strategi terpilih: tandai + isi per kelompok
trip[KOL + "_hilang"] = trip[KOL].isna().astype("int8")
trip[KOL] = trip[KOL].fillna(trip.groupby("jam")[KOL].transform("median"))
trip[KOL] = trip[KOL].fillna(trip[KOL].median())      # jaring pengaman

# Duplikat: periksa dua tingkat
KUNCI = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PULocationID", "DOLocationID", "total_amount"]
dup_penuh = trip.duplicated().sum()
dup_kunci = trip.duplicated(subset=KUNCI).sum()
terdampak = trip.duplicated(subset=KUNCI, keep=False).sum()
print(f"duplikat penuh: {dup_penuh:,} | duplikat pada kunci: {dup_kunci:,} | baris terdampak (semua salinan): {terdampak:,}")

sebelum = len(trip)
trip = trip.drop_duplicates(subset=KUNCI, keep="first").reset_index(drop=True)
print(f"{sebelum:,} -> {len(trip):,} baris")
susut.append(("hapus duplikat (K-7)", len(trip)))

persen hilang: 2.339


,strategi,n,mean,median,std
0,0_asli(dengan NaN),3066766,1.3625,1.0,0.8961
1,1_dibuang,2995023,1.3625,1.0,0.8961
2,2_isi_median,3066766,1.3541,1.0,0.8873
3,3_isi_modus,3066766,1.3541,1.0,0.8873
4,4_median_perjam,3066766,1.3541,1.0,0.8873


duplikat penuh: 0 | duplikat pada kunci: 1 | baris terdampak (semua salinan): 2
3,066,766 -> 3,066,765 baris


## K-8. Outlier, Join, dan Penggabungan Berbasis Waktu


In [10]:
def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def ringkas_outlier(df, kolom):
    hasil = []
    for kol in kolom:
        s = df[kol].dropna()
        lo, hi = batas_iqr(s)
        z = (s - s.mean()) / s.std()
        hasil.append({
            "kolom": kol, "batas_bawah": round(lo, 2), "batas_atas": round(hi, 2),
            "outlier_iqr": int(((s < lo) | (s > hi)).sum()),
            "outlier_z3": int((z.abs() > 3).sum()), "p99": round(s.quantile(0.99), 2),
        })
    return pd.DataFrame(hasil)

tab_outlier = ringkas_outlier(trip, ["trip_distance", "durasi_menit", "total_amount"])
display(tab_outlier)

layak = (trip["trip_distance"].between(0.01, 100)
         & trip["durasi_menit"].between(1, 180)
         & (trip["total_amount"] > 0))
bersih = trip.loc[layak].copy()

lo, hi = batas_iqr(bersih["total_amount"])
bersih["tarif_ekstrem"] = (bersih["total_amount"] > hi).astype("int8")

batas_atas = bersih["trip_distance"].quantile(0.995)
bersih["trip_distance_capped"] = bersih["trip_distance"].clip(upper=batas_atas)

print(f"{len(trip):,} -> {len(bersih):,} baris "
      f"({100*(1-len(bersih)/len(trip)):.2f}% dibuang)")
susut.append(("filter kelayakan (K-8)", len(bersih)))

# --- Join 1: tabel zona ---
print("kunci zona unik?", zona["LocationID"].is_unique)          # WAJIB diperiksa
n0 = len(bersih)
bersih = ukur("join zona", lambda: bersih.merge(
    zona[["LocationID", "Borough", "Zone"]]
        .rename(columns={"Borough": "borough_naik", "Zone": "zona_naik"}),
    left_on="PULocationID", right_on="LocationID",
    how="left", validate="many_to_one"))
bersih = bersih.drop(columns="LocationID")
print(f"baris {n0:,} -> {len(bersih):,} (harus sama)")
m_tanpa_zona = bersih["borough_naik"].isna().to_numpy()
print("tanpa pasangan zona/borough:", int(m_tanpa_zona.sum()))

# --- Join 2: tabel pembayaran dari basis data ---
n0 = len(bersih)
bersih = ukur("join tarif", lambda: bersih.merge(
    tarif_referensi[["payment_type", "nama_pembayaran"]],
    on="payment_type", how="left", validate="many_to_one"))
print(f"baris {n0:,} -> {len(bersih):,}")
print("tanpa pasangan pembayaran:", int(bersih["nama_pembayaran"].isna().sum()))

# --- Join 3: cuaca, berdasarkan jam ---
bersih["jam_mulai"] = bersih["tpep_pickup_datetime"].dt.floor("h").astype("datetime64[ns]")
n0 = len(bersih)
bersih = ukur("join cuaca", lambda: bersih.merge(cuaca, on="jam_mulai", how="left", validate="many_to_one"))
print(f"baris {n0:,} -> {len(bersih):,}")
m_tanpa_cuaca = bersih["suhu_c"].isna().to_numpy()
print("tanpa data cuaca:", int(m_tanpa_cuaca.sum()))

# label eksplisit setelah dihitung
bersih["borough_naik"] = bersih["borough_naik"].fillna("Tidak diketahui")
bersih["nama_pembayaran"] = bersih["nama_pembayaran"].fillna("Tidak diketahui")
susut.append(("setelah join (K-8)", len(bersih)))

,kolom,batas_bawah,batas_atas,outlier_iqr,outlier_z3,p99
0,trip_distance,-2.35,6.74,390244,67,20.06
1,durasi_menit,-9.66,35.08,170615,3175,57.25
2,total_amount,-4.55,48.65,371616,87248,101.94


3,066,765 -> 2,986,910 baris (2.60% dibuang)
kunci zona unik? True
[join zona] 0.67 s
baris 2,986,910 -> 2,986,910 (harus sama)
tanpa pasangan zona/borough: 340
[join tarif] 0.55 s
baris 2,986,910 -> 2,986,910
tanpa pasangan pembayaran: 64106
[join cuaca] 0.85 s
baris 2,986,910 -> 2,986,910
tanpa data cuaca: 37


## K-9. Rekayasa Fitur: Encoding dan Scaling Tanpa Kebocoran


In [11]:
bersih["hari_minggu"] = bersih["tpep_pickup_datetime"].dt.dayofweek
bersih["akhir_pekan"] = (bersih["hari_minggu"] >= 5).astype("int8")
bersih["kecepatan_mph"] = (bersih["trip_distance"] / (bersih["durasi_menit"] / 60)).round(2)
bersih["tarif_per_mil"] = (bersih["total_amount"] / bersih["trip_distance"]).round(3)
bersih["hujan"] = (bersih["hujan_mm"].fillna(0) > 0.1).astype("int8")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

FITUR_NUM = ["trip_distance_capped", "durasi_menit", "suhu_c"]
FITUR_KAT = ["nama_pembayaran", "borough_naik"]
contoh = bersih.dropna(subset=FITUR_NUM + FITUR_KAT).sample(
    n=min(200_000, len(bersih)), random_state=42)
latih, uji = train_test_split(contoh, test_size=0.2, random_state=42)

skala = StandardScaler().fit(latih[FITUR_NUM])          # fit HANYA di data latih
latih_num = skala.transform(latih[FITUR_NUM])
uji_num = skala.transform(uji[FITUR_NUM])               # transform saja
print("mean latih (harus ~0):", latih_num.mean(axis=0).round(3))
print("mean uji (tidak harus 0):", uji_num.mean(axis=0).round(3))

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(latih[FITUR_KAT])
latih_kat = enc.transform(latih[FITUR_KAT])
print("jumlah kolom hasil one-hot:", latih_kat.shape[1])
print("nama kolom:", enc.get_feature_names_out()[:6], "...")

mean latih (harus ~0): [0. 0. 0.]
mean uji (tidak harus 0): [-0.006 -0.004  0.005]
jumlah kolom hasil one-hot: 13
nama kolom: ['nama_pembayaran_Gratis' 'nama_pembayaran_Kartu kredit'
 'nama_pembayaran_Sengketa' 'nama_pembayaran_Tidak diketahui'
 'nama_pembayaran_Tunai' 'borough_naik_Bronx'] ...


## K-10. Pipeline, Validasi, dan Parquet Berpartisi


In [12]:
KONTRAK = {
    "tpep_pickup_datetime": "datetime64[ns]", "trip_distance": "float64",
    "durasi_menit": "float64", "total_amount": "float64",
    "borough_naik": "object", "nama_pembayaran": "object",
}

def _cocok(dtype, tipe):
    d = str(dtype)
    if tipe == "object":
        return d in ("object", "str", "string") or d.startswith("category")
    return d.startswith(tipe.split("[")[0])

def validasi(df, kontrak=KONTRAK):
    masalah = []
    for kol, tipe in kontrak.items():
        if kol not in df.columns:
            masalah.append(f"kolom hilang: {kol}")
        elif not _cocok(df[kol].dtype, tipe):
            masalah.append(f"tipe {kol}: {df[kol].dtype} != {tipe}")
    if df.empty:
        masalah.append("DataFrame kosong")
    elif len(df) != len(df.drop_duplicates(subset=KUNCI)):
        masalah.append("masih ada duplikat pada kunci")
    if masalah:
        raise AssertionError("Validasi gagal:\n- " + "\n- ".join(masalah))
    print(f"Validasi lulus: {len(df):,} baris x {df.shape[1]} kolom")
    return df

validasi(bersih)

def pipeline(path_trip, path_zona, df_cuaca, df_tarif, awal=None, akhir=None, df_libur=None):
    """Satu fungsi, idempoten: input mentah -> DataFrame tervalidasi."""
    df = pd.read_parquet(path_trip, columns=KOLOM)
    z = pd.read_csv(path_zona)
    if awal is not None:
        df = df[(df["tpep_pickup_datetime"] >= awal) & (df["tpep_pickup_datetime"] < akhir)].copy()
    df["durasi_menit"] = ((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
                          .dt.total_seconds() / 60)
    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(
        df.groupby("jam")["passenger_count"].transform("median"))
    df = df.drop_duplicates(subset=KUNCI)
    df = df[df["trip_distance"].between(0.01, 100)
            & df["durasi_menit"].between(1, 180)
            & (df["total_amount"] > 0)]
    df = (df.merge(z[["LocationID", "Borough"]]
                   .rename(columns={"Borough": "borough_naik"}),
                   left_on="PULocationID", right_on="LocationID",
                   how="left", validate="many_to_one")
            .drop(columns="LocationID")
            .merge(df_tarif[["payment_type", "nama_pembayaran"]],
                   on="payment_type", how="left", validate="many_to_one"))
    df["borough_naik"] = df["borough_naik"].fillna("Tidak diketahui")
    df["nama_pembayaran"] = df["nama_pembayaran"].fillna("Tidak diketahui")
    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h").astype("datetime64[ns]")
    cw = df_cuaca.assign(jam_mulai=df_cuaca["jam_mulai"].astype("datetime64[ns]"))
    df = df.merge(cw, on="jam_mulai", how="left", validate="many_to_one")
    df["tanggal"] = df["tpep_pickup_datetime"].dt.date.astype(str)
    if df_libur is not None:
        df = df.merge(df_libur, on="tanggal", how="left", validate="many_to_one")
        df["hari_libur"] = df["nama_libur"].notna().astype("int8")
    return validasi(df.reset_index(drop=True))

def tulis_bersih(df, out):
    """Idempoten: hapus keluaran lama, lalu tulis ulang."""
    if os.path.isdir(out):
        shutil.rmtree(out)
    df.to_parquet(out, partition_cols=["borough_naik"], index=False)

def daftar_berkas(out):
    return sorted(os.path.relpath(p, out) for p in glob.glob(os.path.join(out, "**", "*.parquet"), recursive=True))

hasil = ukur("pipeline penuh", lambda: pipeline(PATH_TRIP, PATH_ZONA, cuaca, tarif_referensi))
susut.append(("hasil pipeline (K-10)", len(hasil)))

OUT = os.path.join(DIR_KURASI, "trips_bersih")
ukur("tulis parquet berpartisi", lambda: tulis_bersih(hasil, OUT))
print("partisi yang terbentuk:", sorted(os.listdir(OUT))[:5], "...")

# Bukti idempotensi: jalankan ulang pipeline dan penulisan
def sidik(d):
    return len(d), round(float(d["total_amount"].sum()), 2)

ulang = pipeline(PATH_TRIP, PATH_ZONA, cuaca, tarif_referensi)
berkas_1 = daftar_berkas(OUT)
tulis_bersih(ulang, OUT)
berkas_2 = daftar_berkas(OUT)
print("sidik pipeline :", sidik(hasil), sidik(ulang))
print("berkas parquet :", len(berkas_1), len(berkas_2))
print("idempoten?", sidik(hasil) == sidik(ulang) and berkas_1 == berkas_2)

# Artefak yang dikumpulkan
laporan.to_csv(f"{DIR_SIMPAN}/laporan_kualitas.csv", index=False)
pd.DataFrame(lineage).to_csv(f"{DIR_SIMPAN}/lineage.csv", index=False)
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/pengukuran_kinerja.csv", index=False)
shutil.make_archive(f"{DIR_SIMPAN}/trips_bersih", "zip", OUT)
print("artefak tersimpan di:", DIR_SIMPAN)

Validasi lulus: 2,986,910 baris x 26 kolom
Validasi lulus: 2,986,910 baris x 18 kolom
[pipeline penuh] 14.24 s
[tulis parquet berpartisi] 3.47 s
partisi yang terbentuk: ['borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens'] ...
Validasi lulus: 2,986,910 baris x 18 kolom
sidik pipeline : (2986910, 81621442.46) (2986910, 81621442.46)
berkas parquet : 8 8
idempoten? False
artefak tersimpan di: /content/drive/MyDrive/BigData/Praktikum3


## K-11. Pra-pemrosesan Setara dengan PySpark (Tambahan)


In [13]:
!pip install -q pyspark==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 16.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [14]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("BD-P03").master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

sdf = spark.read.parquet(PATH_TRIP).select(*KOLOM)
zona_spark = zona[["LocationID", "Borough"]].copy()
zona_spark["Borough"] = zona_spark["Borough"].fillna("Tidak diketahui")
szona = spark.createDataFrame(zona_spark)

sbersih = (sdf
    .withColumn("durasi_menit",
                (F.col("tpep_dropoff_datetime").cast("timestamp").cast("long")
                 - F.col("tpep_pickup_datetime").cast("timestamp").cast("long")) / 60)
    .filter(F.col("trip_distance").between(0.01, 100))
    .filter(F.col("durasi_menit").between(1, 180))
    .filter(F.col("total_amount") > 0)
    .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime",
                     "PULocationID", "DOLocationID", "total_amount"])
    .join(F.broadcast(szona), F.col("PULocationID") == F.col("LocationID"), "left")
    .drop("LocationID"))

n_spark = ukur("spark: hitung baris bersih", lambda: sbersih.count())
print("baris bersih Spark :", f"{n_spark:,}")
print("baris bersih pandas:", f"{len(hasil):,}", "| selisih:", n_spark - len(hasil))

sbersih.groupBy("Borough").count().orderBy(F.desc("count")).show(10, False)
sbersih.write.mode("overwrite").partitionBy("Borough").parquet("/content/lapisan_terkurasi/trips_spark")

sbersih.explain()
try:
    rencana = sbersih._jdf.queryExecution().executedPlan().toString()
    print("BroadcastHashJoin pada rencana eksekusi?", "BroadcastHashJoin" in rencana)
except Exception as e:
    print("tidak bisa membaca rencana eksekusi:", e)
spark.stop()

[spark: hitung baris bersih] 26.77 s
baris bersih Spark : 2,986,910
baris bersih pandas: 2,986,910 | selisih: 0
+---------------+-------+
|Borough        |count  |
+---------------+-------+
|Manhattan      |2659609|
|Queens         |270315 |
|Unknown        |37609  |
|Brooklyn       |15724  |
|Bronx          |3074   |
|Tidak diketahui|340    |
|Staten Island  |211    |
|EWR            |28     |
+---------------+-------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#245, trip_distance#247, PULocationID#7L, DOLocationID#8L, payment_type#249L, fare_amount#251, tip_amount#253, total_amount#16, durasi_menit#255, Borough#50]
   +- BroadcastHashJoin [PULocationID#7L], [LocationID#49L], LeftOuter, BuildRight, false
      :- HashAggregate(keys=[DOLocationID#8L, tpep_dropoff_datetime#2, PULocationID#7L, total_amount#16, tpep_pickup_datetime#1], functions=[first(passenger_count#3, false), first(trip_distance#4

## P. Studi Kasus — Tabel Analitik untuk Tim Penetapan Harga Dinamis


In [15]:
tabel_analitik = (bersih
    .dropna(subset=["suhu_c"])
    .assign(jam_mulai=bersih["tpep_pickup_datetime"].dt.floor("h"))
    .groupby(["borough_naik", "jam_mulai"], observed=True)
    .agg(jumlah_perjalanan=("total_amount", "size"),
         rata_tarif_per_mil=("tarif_per_mil", "median"),
         rata_kecepatan=("kecepatan_mph", "median"),
         suhu_c=("suhu_c", "first"), hujan=("hujan", "max"))
    .reset_index())

tabel_analitik = tabel_analitik[tabel_analitik["jumlah_perjalanan"] >= 30]   # ambang minimum
tabel_analitik.to_parquet(f"{DIR_SIMPAN}/tabel_analitik.parquet", index=False)

banding_hujan = (tabel_analitik
    .groupby(["borough_naik", "hujan"], observed=True)["rata_tarif_per_mil"]
    .median().unstack())
banding_hujan.columns = ["tanpa_hujan" if c == 0 else "hujan" for c in banding_hujan.columns]
if {"tanpa_hujan", "hujan"} <= set(banding_hujan.columns):
    banding_hujan["selisih_persen"] = ((banding_hujan["hujan"] / banding_hujan["tanpa_hujan"] - 1) * 100).round(2)
display(banding_hujan)
print("jumlah baris tabel analitik:", len(tabel_analitik))

,tanpa_hujan,hujan,selisih_persen
borough_naik,,,
Brooklyn,6.9430,7.6550,10.25
Manhattan,10.5270,11.4000,8.29
Queens,5.4530,5.6365,3.37
Unknown,8.7255,9.8860,13.30


jumlah baris tabel analitik: 2094


### Jawaban dugaan tim pricing
Baca kolom `selisih_persen` di tabel di atas: nilai positif berarti median tarif per mil saat hujan **lebih tinggi** daripada tanpa hujan untuk borough itu. Nyatakan per borough apakah dugaan didukung, dan sebutkan borough yang tidak punya cukup jam hujan (kolom `hujan` kosong/NaN) sebagai **tidak dapat disimpulkan**.

### Batasan yang harus dinyatakan
1. **Median, bukan rerata**, karena distribusi tarif per mil sangat miring; median tidak menangkap perubahan pada ekor (perjalanan mahal).
2. **Cuaca dari satu titik** (koordinat pusat New York) untuk seluruh kota, sehingga hujan lokal di borough lain bisa tidak terwakili.
3. **Korelasi bukan sebab-akibat**: hujan bisa bersamaan dengan jam sibuk, hari kerja, atau musim, sehingga naiknya tarif per mil bisa berasal dari macet (waktu tunggu argo) bukan hujan itu sendiri.
4. **Cakupan satu bulan** (Januari 2023) dan ambang minimum 30 perjalanan per borough-jam membuat borough kecil (Staten Island, Bronx, EWR) jarang lolos; hasilnya tidak untuk digeneralisasi.
5. **Tarif per mil** bergantung pada jarak; perjalanan pendek punya tarif per mil tinggi karena biaya dasar, sehingga campuran jarak yang berbeda saat hujan dapat menyesatkan.

### Pembaruan otomatis bulanan
Jadwalkan (Airflow/cron/Colab terjadwal) pemanggilan `pipeline_inkremental(bulan)` (lihat Latihan 6) setiap awal bulan untuk bulan yang baru terbit. Fungsi itu mengunduh berkas bulan tersebut dengan `unduh_aman`, mengambil cuaca bulan yang sama, menjalankan `pipeline` (dengan `validasi` yang gagal berisik), lalu menimpa **hanya partisi `bulan=YYYY-MM`** sehingga aman dijalankan ulang (idempoten) dan tidak menduplikasi bulan lain. Setelah itu tabel analitik dihitung ulang dari seluruh direktori berpartisi, `lineage.csv` diperbarui (sumber, waktu ambil, jumlah baris, versi kode) dan `laporan_kualitas.csv` bulan baru dibandingkan dengan bulan sebelumnya untuk memicu peringatan bila ada aturan yang tingkat kegagalannya melonjak.

## Q. Latihan

### Latihan 1 — Kecepatan unduh dan bukti cache


In [16]:
if os.path.exists(PATH_ZONA) and SUMBER.get("zona") == "nyata":
    os.remove(PATH_ZONA)
    print("--- panggilan 1 ---"); unduh_aman(URL_ZONA, PATH_ZONA)
    print("--- panggilan 2 ---"); unduh_aman(URL_ZONA, PATH_ZONA)
else:
    print("Zona sintetis/tidak ada: latihan ini butuh internet. Panggilan ke-2 di bawah tetap membuktikan cache:")
    unduh_aman(URL_ZONA, PATH_ZONA)

--- panggilan 1 ---
unduh taxi_zone_lookup.csv: 0.0 MB dalam 0.1s (0.17 MB/s)
--- panggilan 2 ---
cache ditemukan: taxi_zone_lookup.csv


### Latihan 2 — Dua aturan kualitas baru


In [17]:
mentah = pd.read_parquet(PATH_TRIP, columns=KOLOM)
aturan_baru = {
    "validitas: tip_amount 0-200": mentah["tip_amount"].between(0, 200),
    "konsistensi: tip <= total_amount": mentah["tip_amount"] <= mentah["total_amount"],
}
laporan_baru = pd.DataFrame([
    {"aturan": n, "lulus": int(m.sum()), "gagal": int((~m).sum()),
     "persen_gagal": round(100 * (~m).mean(), 3)} for n, m in aturan_baru.items()])
laporan_baru

,aturan,lulus,gagal,persen_gagal
0,validitas: tip_amount 0-200,3066534,232,0.008
1,konsistensi: tip <= total_amount,3041562,25204,0.822


### Latihan 3 — Join lokasi tujuan dan 10 pasangan borough asal–tujuan tersibuk

In [18]:
zona_do = (zona[["LocationID", "Borough"]]
           .assign(Borough=lambda d: d["Borough"].fillna("Tidak diketahui"))
           .rename(columns={"Borough": "borough_turun"}))
n0 = len(hasil)
od = hasil.merge(zona_do, left_on="DOLocationID", right_on="LocationID",
                 how="left", validate="many_to_one").drop(columns="LocationID")
print(f"baris {n0:,} -> {len(od):,} (harus sama)")
od["borough_turun"] = od["borough_turun"].fillna("Tidak diketahui")
(od.groupby(["borough_naik", "borough_turun"]).size()
   .sort_values(ascending=False).head(10).rename("jumlah_perjalanan").reset_index())

baris 2,986,910 -> 2,986,910 (harus sama)


,borough_naik,borough_turun,jumlah_perjalanan
0,Manhattan,Manhattan,2491566
1,Queens,Manhattan,155261
2,Manhattan,Queens,82354
3,Manhattan,Brooklyn,62986
4,Queens,Queens,59146
5,Queens,Brooklyn,42187
6,Unknown,Manhattan,18432
7,Unknown,Unknown,13562
8,Manhattan,Bronx,8865
9,Brooklyn,Brooklyn,7910


### Latihan 4 — `MinMaxScaler` vs `StandardScaler`

In [19]:
from sklearn.preprocessing import MinMaxScaler
mm = MinMaxScaler().fit(latih[FITUR_NUM])
rentang = pd.DataFrame({
    "std_min_latih": latih_num.min(axis=0).round(3), "std_max_latih": latih_num.max(axis=0).round(3),
    "mm_min_latih": mm.transform(latih[FITUR_NUM]).min(axis=0).round(3),
    "mm_max_latih": mm.transform(latih[FITUR_NUM]).max(axis=0).round(3),
    "mm_min_uji": mm.transform(uji[FITUR_NUM]).min(axis=0).round(3),
    "mm_max_uji": mm.transform(uji[FITUR_NUM]).max(axis=0).round(3),
}, index=FITUR_NUM)
rentang

,std_min_latih,std_max_latih,mm_min_latih,mm_max_latih,mm_min_uji,mm_max_uji
trip_distance_capped,-0.798,4.271,0.0,1.0,0.0,1.000
durasi_menit,-1.238,14.923,0.0,1.0,0.0,0.929
suhu_c,-2.284,3.354,0.0,1.0,0.0,1.000


**Jawaban:** StandardScaler menghasilkan rerata 0 dan std 1 (rentang tak terbatas); MinMaxScaler memaksa rentang [0, 1] pada data latih, dan data uji bisa keluar dari [0, 1] bila ada nilai di luar rentang latih. Perbedaan ini penting untuk algoritma berbasis jarak/gradien (k-NN, SVM, k-means, regresi/jaringan saraf) karena skala fitur memengaruhi jarak dan konvergensi; MinMax sensitif terhadap outlier. Tidak penting untuk model berbasis pohon (decision tree, random forest, gradient boosting) yang hanya memakai urutan nilai untuk memecah.

### Latihan 5 — Bukti `validate="many_to_one"` bekerja

In [20]:
zona_ganda = pd.concat([zona, zona.iloc[[0]]], ignore_index=True)   # kunci LocationID=1 ganda
print("kunci unik pada tabel ganda?", zona_ganda["LocationID"].is_unique)
try:
    hasil[["PULocationID"]].head(1000).merge(
        zona_ganda[["LocationID", "Borough"]], left_on="PULocationID", right_on="LocationID",
        how="left", validate="many_to_one")
    print("TIDAK ada error (seharusnya ada) - cek apakah LocationID=1 muncul pada sampel")
except Exception as e:
    print(type(e).__name__, "->", e)
# `zona` asli tidak diubah (kita hanya membuat salinan `zona_ganda`)
print("tabel zona asli kembali utuh:", zona["LocationID"].is_unique, len(zona))

kunci unik pada tabel ganda? False
MergeError -> Merge keys are not unique in right dataset; not a many-to-one merge
tabel zona asli kembali utuh: True 265


Catatan: bila sampel 1.000 baris tidak memuat `PULocationID == 1`, pandas hanya memeriksa keunikan kunci pada tabel kanan (bukan hanya yang terpakai), sehingga `MergeError` tetap muncul.

### Latihan 6 — `pipeline_inkremental(bulan)` (dan Tugas 1, 3)


In [21]:
def ambil_libur(tahun, negara="US"):
    url = f"https://date.nager.at/api/v3/PublicHolidays/{tahun}/{negara}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        df = pd.DataFrame(r.json())
        df = df.rename(columns={"date": "tanggal", "name": "nama_libur"})[["tanggal", "nama_libur"]]
        df = df.drop_duplicates(subset="tanggal").reset_index(drop=True)
        catat_sumber(f"libur_{tahun}", url, len(df), "Nager.Date PublicHolidays")
        return df
    except Exception as e:
        print(f"PERINGATAN: API libur gagal ({e}); memakai daftar manual 2023.")
        df = pd.DataFrame({"tanggal": ["2023-01-01", "2023-01-02", "2023-01-16", "2023-07-04"],
                           "nama_libur": ["New Year's Day", "New Year's Day (observed)",
                                          "Martin Luther King Jr. Day", "Independence Day"]})
        catat_sumber(f"libur_{tahun}", "manual", len(df), "PENGGANTI - daftar manual")
        return df

libur_2023 = ambil_libur(2023)
libur_2023.head()

[lineage] libur_2023: 14 baris


,tanggal,nama_libur
0,2023-01-02,New Year's Day
1,2023-01-16,"Martin Luther King, Jr. Day"
2,2023-02-12,Lincoln's Birthday
3,2023-02-20,Presidents Day
4,2023-04-07,Good Friday


In [22]:
DIR_INK = os.path.join(DIR_KURASI, "trips_bersih_2bulan")

def pipeline_inkremental(bulan, paksa=False):
    out = os.path.join(DIR_INK, f"bulan={bulan}")
    if daftar_berkas(out) and not paksa:
        print(f"[{bulan}] sudah ada -> dilewati")
        return 0
    awal, akhir = rentang_bulan(bulan)
    path = pastikan_trip(bulan)
    cw, sumber_cw = cuaca_bulan(bulan)
    SUMBER[f"cuaca_{bulan}"] = sumber_cw
    df = pipeline(path, PATH_ZONA, cw, tarif_referensi, awal, akhir, ambil_libur(int(bulan[:4])))
    tulis_bersih(df, out)
    n = len(daftar_berkas(out))
    print(f"[{bulan}] ditulis: {len(df):,} baris, {n} berkas")
    return n

# Tugas 1: dua bulan ke direktori berpartisi yang sama
for b in ["2023-01", "2023-07"]:
    ukur(f"pipeline_inkremental {b}", lambda b=b: pipeline_inkremental(b))

# Bukti idempotensi (Latihan 6): panggilan kedua tidak menghasilkan berkas baru
sebelum_ = daftar_berkas(DIR_INK)
baru = [pipeline_inkremental(b) for b in ["2023-01", "2023-07"]]
sesudah_ = daftar_berkas(DIR_INK)
print("berkas baru pada panggilan ke-2:", sum(baru), "| daftar berkas sama:", sebelum_ == sesudah_)

# Tugas 1: buktikan tidak ada duplikasi antar-bulan
cek = pd.read_parquet(DIR_INK, columns=KUNCI + ["bulan"])
cek["bulan"] = cek["bulan"].astype(str)
print("baris per bulan:\n", cek["bulan"].value_counts().sort_index())
print("duplikat pada kunci (seluruh direktori):", int(cek.duplicated(subset=KUNCI).sum()))
print("bulan pickup sesuai partisi?",
      bool((cek["tpep_pickup_datetime"].dt.strftime("%Y-%m") == cek["bulan"]).all()))
del cek; gc.collect()

cache ditemukan: yellow_tripdata_2023-01.parquet
[lineage] libur_2023: 14 baris
Validasi lulus: 2,986,873 baris x 20 kolom
[2023-01] ditulis: 2,986,873 baris, 8 berkas
[pipeline_inkremental 2023-01] 35.07 s
unduh yellow_tripdata_2023-07.parquet: 46.1 MB dalam 0.3s (133.17 MB/s)
[lineage] libur_2023: 14 baris
Validasi lulus: 2,817,607 baris x 20 kolom
[2023-07] ditulis: 2,817,607 baris, 8 berkas
[pipeline_inkremental 2023-07] 20.01 s
[2023-01] sudah ada -> dilewati
[2023-07] sudah ada -> dilewati
berkas baru pada panggilan ke-2: 0 | daftar berkas sama: True
baris per bulan:
 bulan
2023-01    2986873
2023-07    2817607
Name: count, dtype: int64
duplikat pada kunci (seluruh direktori): 0
bulan pickup sesuai partisi? True


176

Tugas Praktikum

In [ ]:
def laporan_bulan(bulan):
    path = pastikan_trip(bulan)
    awal, akhir = rentang_bulan(bulan)
    df = pd.read_parquet(path, columns=KOLOM)
    lap = hitung_laporan(df, awal, akhir)
    del df; gc.collect()
    return lap

lap_jan, lap_jul = laporan_bulan("2023-01"), laporan_bulan("2023-07")
komparatif = pd.DataFrame({
    "gagal_jan_%": lap_jan.set_index("aturan")["persen_gagal"],
    "gagal_jul_%": lap_jul.set_index("aturan")["persen_gagal"],
})
komparatif["selisih_poin"] = (komparatif["gagal_jul_%"] - komparatif["gagal_jan_%"]).round(3)
komparatif["berubah_signifikan"] = komparatif["selisih_poin"].abs() >= 0.5   # ambang: 0,5 poin persen
komparatif = komparatif.sort_values("selisih_poin", key=abs, ascending=False)
komparatif.to_csv(f"{DIR_SIMPAN}/laporan_kualitas_komparatif.csv")
komparatif

Kamus Data md

In [25]:
hasil_j = pd.read_parquet(DIR_INK)   # dataset akhir dua bulan (dengan kolom bulan & borough_naik dari partisi)
META = {
 "tpep_pickup_datetime": ("waktu", "TLC parquet", "tidak boleh kosong; harus dalam bulan berkas", "tidak ada yang hilang; baris di luar periode dibuang"),
 "tpep_dropoff_datetime": ("waktu", "TLC parquet", "harus setelah waktu naik", "menentukan durasi; dibuang bila durasi tak valid"),
 "passenger_count": ("orang", "TLC parquet", "bilangan bulat 0-9 (wajar)", "diisi median per jam"),
 "trip_distance": ("mil", "TLC parquet", "0,01-100", "baris di luar rentang dibuang"),
 "PULocationID": ("id zona", "TLC parquet", "harus ada di taxi_zone_lookup", "kunci join zona naik"),
 "DOLocationID": ("id zona", "TLC parquet", "harus ada di taxi_zone_lookup", "kunci join zona turun (latihan 3)"),
 "payment_type": ("kode", "TLC parquet", "1-6 sesuai tarif_referensi", "tanpa pasangan -> nama 'Tidak diketahui'"),
 "fare_amount": ("USD", "TLC parquet", ">= 0 dan <= total_amount", "tidak diisi"),
 "tip_amount": ("USD", "TLC parquet", "0-200", "tidak diisi; tip 0 untuk tunai adalah nilai yang sah"),
 "total_amount": ("USD", "TLC parquet", "> 0", "baris <= 0 dibuang; tarif ekstrem hanya ditandai"),
 "durasi_menit": ("menit", "turunan", "1-180", "dihitung dari waktu turun - naik"),
 "jam": ("jam 0-23", "turunan", "0-23", "-"),
 "borough_naik": ("kategori", "taxi_zone_lookup (join)", "salah satu borough atau 'Tidak diketahui'", "NaN -> 'Tidak diketahui'"),
 "nama_pembayaran": ("kategori", "SQLite tarif_referensi (join)", "salah satu dari tabel referensi", "NaN -> 'Tidak diketahui'"),
 "jam_mulai": ("waktu (per jam)", "turunan", "waktu naik dibulatkan ke bawah per jam", "kunci join cuaca"),
 "suhu_c": ("derajat C", "Open-Meteo (join)", "wajar -40..50", "NaN bila tak ada data cuaca; tidak diisi"),
 "hujan_mm": ("mm/jam", "Open-Meteo (join)", ">= 0", "NaN bila tak ada data cuaca; dianggap 0 pada penanda hujan"),
 "tanggal": ("YYYY-MM-DD", "turunan", "tanggal waktu naik", "kunci join hari libur"),
 "nama_libur": ("teks", "Nager.Date (join)", "nama libur atau kosong", "kosong = bukan hari libur"),
 "hari_libur": ("0/1", "turunan", "1 jika tanggal libur nasional", "-"),
 "bulan": ("YYYY-MM", "kolom partisi", "sesuai direktori bulan=", "-"),
}
baris_md = ["# Kamus Data — Dataset Akhir (trips_bersih_2bulan)", "",
            "| Kolom | Tipe | Satuan | Sumber | Aturan validitas | Penanganan nilai hilang |",
            "|---|---|---|---|---|---|"]
for kol in hasil_j.columns:
    sat, sumb, aturan_v, hilang = META.get(kol, ("(lengkapi)",) * 4)
    baris_md.append(f"| `{kol}` | {hasil_j[kol].dtype} | {sat} | {sumb} | {aturan_v} | {hilang} |")

baris_md += ["", "# Kamus Data — Tabel Analitik Studi Kasus (tabel_analitik.parquet)", "",
             "| Kolom | Definisi | Satuan | Cara penghitungan |", "|---|---|---|---|",
             "| `borough_naik` | Borough tempat penjemputan | kategori | hasil join zona |",
             "| `jam_mulai` | Awal jam pengelompokan | waktu | waktu naik dibulatkan ke bawah per jam |",
             "| `jumlah_perjalanan` | Banyak perjalanan pada borough-jam | perjalanan | `size()`; kelompok < 30 dibuang |",
             "| `rata_tarif_per_mil` | Median tarif per mil | USD/mil | median dari `total_amount / trip_distance` |",
             "| `rata_kecepatan` | Median kecepatan | mph | median dari `trip_distance / (durasi_menit/60)` |",
             "| `suhu_c` | Suhu jam tersebut | derajat C | nilai pertama pada jam (satu titik untuk seluruh kota) |",
             "| `hujan` | Penanda hujan | 0/1 | 1 bila `hujan_mm > 0,1` pada jam itu |"]
with open(f"{DIR_SIMPAN}/kamus_data.md", "w", encoding="utf-8") as f:
    f.write("\n".join(baris_md))
print("\n".join(baris_md[:8]), "\n...\nkamus_data.md tersimpan; kolom '(lengkapi)' = tambahkan manual bila ada.")
del hasil_j; gc.collect()

# Kamus Data — Dataset Akhir (trips_bersih_2bulan)

| Kolom | Tipe | Satuan | Sumber | Aturan validitas | Penanganan nilai hilang |
|---|---|---|---|---|---|
| `tpep_pickup_datetime` | datetime64[us] | waktu | TLC parquet | tidak boleh kosong; harus dalam bulan berkas | tidak ada yang hilang; baris di luar periode dibuang |
| `tpep_dropoff_datetime` | datetime64[us] | waktu | TLC parquet | harus setelah waktu naik | menentukan durasi; dibuang bila durasi tak valid |
| `passenger_count` | float64 | orang | TLC parquet | bilangan bulat 0-9 (wajar) | diisi median per jam |
| `trip_distance` | float64 | mil | TLC parquet | 0,01-100 | baris di luar rentang dibuang | 
...
kamus_data.md tersimpan; kolom '(lengkapi)' = tambahkan manual bila ada.


16

Tugas 5 Diagram Liniage

In [ ]:
from matplotlib.patches import FancyBboxPatch

s = dict(susut)
def kotak(ax, x, y, w, h, teks, warna):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03", fc=warna, ec="#333"))
    ax.text(x + w / 2, y + h / 2, teks, ha="center", va="center", fontsize=8)
def panah(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#333"))

fig, ax = plt.subplots(figsize=(15, 6.5))
ax.set_xlim(0, 15); ax.set_ylim(0, 7); ax.axis("off")
ax.set_title("Data lineage — Praktikum 3 (Yellow Taxi + zona + cuaca + SQLite + hari libur)", fontsize=11)

sumber_kotak = ["NYC TLC Parquet\n(unduh_aman, retry)", "Taxi Zone CSV\n(unduh_aman)",
                "Open-Meteo API\n(retry/backoff)", "SQLite tarif_referensi\n(SQLAlchemy)", "Nager.Date API\n(hari libur)"]
for i, t in enumerate(sumber_kotak):
    y = 5.6 - i * 1.25
    kotak(ax, 0.2, y, 2.6, 0.9, t, "#dfe9f5")
    panah(ax, 2.8, y + 0.45, 3.5, 3.4)

n0 = s.get("awal (mentah)", 0)
tahap = [
    (3.5, f"lapisan_mentah/\nn={n0:,}", "#eee"),
    (5.7, f"K-7: isi hilang + dedup\nn={s.get('hapus duplikat (K-7)', 0):,}", "#fff2cc"),
    (7.9, f"K-8: filter kelayakan\nn={s.get('filter kelayakan (K-8)', 0):,}", "#fce5cd"),
    (10.1, f"join zona/tarif/cuaca/libur\nn={s.get('setelah join (K-8)', 0):,} (tetap)", "#d9ead3"),
    (12.3, f"validasi kontrak +\nParquet berpartisi\nn={s.get('hasil pipeline (K-10)', 0):,}", "#cfe2f3"),
]
for i, (x, t, w) in enumerate(tahap):
    kotak(ax, x, 2.9, 1.9, 1.0, t, w)
    if i:
        panah(ax, tahap[i - 1][0] + 1.9, 3.4, x, 3.4)
ax.text(7.9, 2.2, "baris dibuang: durasi/jarak/tarif tidak valid", fontsize=8, ha="center", color="#a00")
ax.text(5.7, 2.2, "baris dibuang: duplikat pada kunci", fontsize=8, ha="center", color="#a00")
plt.tight_layout()
plt.savefig(f"{DIR_SIMPAN}/lineage_diagram.png", dpi=150)
plt.show()